# Slide 21 v2: seedance (hosted API) on the 3D tube stimuli

Still mode: the model gets the last frame before the exit plus one fixed prompt saying the ball is exiting the tube (no direction given), and makes 5 s at 480p.
1 sample(s) per item, items: `all`. Spending cap: about $45 at $0.27/s (estimate; fal bills the real price).

Before running: **Add Input** -> the `slide21-render3d-v2-all` kernel output; **Add-ons -> Secrets** -> `FAL_KEY`; **Internet on**. No GPU needed.
Clips already downloaded are skipped, so re-running after a stop does not pay twice.

In [ ]:
!pip install -q fal-client imageio imageio-ffmpeg

In [ ]:
import base64, io, tarfile, os, sys, json, glob, math, time, subprocess, urllib.request
import numpy as np, imageio.v2 as imageio
from PIL import Image
from kaggle_secrets import UserSecretsClient
os.environ["FAL_KEY"] = UserSecretsClient().get_secret("FAL_KEY")
import fal_client
SRC = "/kaggle/working/src"
tarfile.open(fileobj=io.BytesIO(base64.b64decode("H4sIAAVN92oC/+0825LbxpV+5ld0oPIKkEgMOVeJEbQ7keXIu7bitRQ7VQwX1UOCJDQggKBBDilH/7Rv+54v23PpBhokOLo4TsrOsGqGRF9O3869z4F/5B/9x7dy8yKS06j47Gf59Plz6LvfPzmpf2P5oH88GHwmNp/9Az4rVcoChv/sX/NzfCGWZbyMgsHF4xM4iYuzc//R44vHj087n919fv0fmcpk+zby8+3PNwYS9fnpKX4PLk4v+PvM0P/JoH9x+tng7Pj84nzQPzk7A/o/PjsG+u//I+l/ksjVNLql3dUqLVe/uvO/95ujlSqOruL0KErXIt+Wiyw96TiO82qSFZEoF5FQSTyNescDUUaqHIppFikqX2bTKBHXUZSLyapYx+lcyFkZFVR5JZNEJJFc68bl6ir6907n9U0mCpA12apUXXGVlQuRYw+oFWoh80jIdCriNF+VBF+4qowB0lqJSRLn3rDTEWIepVEhy0gI9+vXf+p9D9PLvKEoCzm5psGKaMoTiFMRycmi6jElKF2xjKRawfIW2Y1YrqAB9ALA9SeXMLOrKJ0qa03RJi67NEHAFqVEXAoJ/5d5VK6UcLk5NryR22rRDbDQpPS6Is3EVQxdYXGFjOcLLJtAVRHPVnOZ2KBgh2B0AAhNsgIWtJZwHA2gLu91pkpskWa4c3gani+2IhCfVzOUywzOiADQPigf4LxZTecRw/m+95/Pv70Ux7CXM4BEOxeX0RImly1zqdGBjv2+ErCBeRErmGJK5YAvRTQpm1OrFgjVaRmnK1nG0B73UHertk/jkAcVkd+AwqvIZjQZJW5gR3gqBu0asGMlvn7+6pWZH9T7nc4flZxHQ4DKGC5qvid6PUI3JfRXrwfYIgBBj5JyA0/rN1Eu6Zl++RO1htLFailTJejraCpLCWXQBlBPrZJSdTovsEZQjcutog2gOsjatPQX5TLp4kLFs1ffEwXAn8pSD2evkPKm4iYuF0x/cgkovUoiNexMC3mTEnYqMc+gushW8wXjYzzbuh4gqZyUyVYk8XXEJAGwmFSpGyOwFM7NIobzTbLsWnVSWa4KmThissjiSYR7be/vGiaHWzvJgAkqBq2RJS+iWVQU2Cou/c4fUhg6jSLA3nS1zLddOF/Y+TgT7kPzszebwZnPvd+KpSzzJCuT+IoQDofEZ+FmOR6lTDwfWVGnA0iSFUBtxRzQUEXmGY7C/Jwn2ZX5/QZ20vyGIRbmd6Y6syJbwjKSBDAVRiDqxappNJNwbtMYENg0pwUghad5p1MW2yEhpa7Ua/HXx8wD6KkTbSZRXoqvqM3zosiKoRD3YJPkfCmHSPeTDDZTA+KNCcRLQIRO59WLy2+fv4LHkWOoxukKZ/K4T1+DR/x9fEHfKo/xwMbcLfz68nfPv4a+P9Z9h2IPDhQ97v/tfyt48AxfugABQwF8cYEeYVj9etd59vVX34bP/vDyi/DL7y6/ock+ZhK9RwyF8AyYRwEYq0Tf9x8hJskd9isMJwGyncZ4DIg9LrG8AXcGRjkz4ge5rtf53XMY9Yvnv4chz/y+zR3uib8uQKIQH1jIdB79Vcgr2GXoDMQUMFbV7NxVUWnAI/KT2CqJUL3ON1+9DL/9w1cvX+PSTpujGFrSq2uKBcJ5rAQ0lhaLJYivv7v8nk7npG9NHdBiU4vKJSgiyLcjBHEji+nuAHuQO/eY+wiiwCgFss2zGOkTKJVktgCZ7W6Ar02jjWG3WshuYmAEnxsC9zov/vjN5UtCPsJNpAM3kVdREjjM6bqCWQ8RkBQsld1vJs9A7lxHWwGbKhNfDB4/6nuAPZvg2D/rim1wcgLSVhbXURE4XzgoSZIMft774vGjy9PnjtfdHQ+JAfufYu+zwUf2rmY7QMw715vYMt9/E/+VLRKc7wnP98R/fIxDHp/VQ6pPH/JGlpMF4fUtY576/Uc05qP2MY+fn10+usAxx51OB3gUYKychiQGXRZXnuZKJBmB/t/RIyJLjtoPskUf/7mZ8pEK/DeAI7ovEPmDB0jqy6iUPnJNGArQCbBCxesoeF2sIg0fP0uAj418nASw6Ch1c8+rquOZWPrzqHSdmVzGydbxRBAI53LgDBvSfDlypnHhjAGamRM8p0BUAK7RkhY1gvb4I4ynzhg7LalNAUpDkXITvTdEoC6Tp542CI9nqFdVEm1PM2R2476UL1GvSEl9moGYm4L2xKCGwn3dFS+64oeuOPHECkjskY9SiTYaJsTNfKnKbR65ae5Di8E5L6Xoijnoudhq5Pt+V/THXfNzUP88HlPrpVTX0NQtxFMxOO17gDHwuwcCyF8CyS5XSxeheVB90ecBUOcIsMFslSSum8Ch6B0AqKhq5n4qU6/CihgwjRYP0o14souDWqe8BbTYKIaZZunbqMjcZeOUcYyNgkkE4rx5tPfYGsCNTkBWg7kgrgD3BCkpyXYo8ngTJSBQI8mcbRlNY5k20WMDM9zy+FyNg3Wtx61qogmNCbsGLeSVgtawY8uNJ56IY95CXbGliq2uaGLajKD4Cna4bWF6p0cxIuBGjbDxGOYDs4GpbZsFNn5CJ42dlYrGTLpLXD3cgIakgFVb+EqmBaqGJLSaUsDvUDMGAYj5Eg+ZEZrsh2jfXIGzRgcPmAZTKEdEhzZJkt1EgOIETU9kWAsjkIcGwG9rCVSicE1LVFEfbgiHQJXZissBUSHDwrUMxcOBOBI94N2s4IOe09sCyyxY6apMI2CSiqSylsFmJGvrXaMXGlWGOvJUcVSwQ2ifcVzYyUxEMRlMCNWoqzCIkXM8ye/obEAVIAbeFVp9CFl9CKfRXJQZyuDadNUDpCFvvQZkuACiH1eMENeAEJELqFmcwhz1gQOeoNQNBh7TuiakHNGxVjtqtNMI5GiLL0T7Dnj1DJgvsFigaeTVDIElEs6BeYVUsijk1tUHW9dPR9PREHkQojj8b/vcA9xLtlprQp27wifahyVg9EJoAQKnkqGYwc2pmnj28qY/eXlTzbtueAE0fTMCFCJjdM0grGgdHgSnm0fT27bxnsEGIzESdDTQTugCIssigvIYlcxsn0Dchxs9aWBzOGmcPO64fyYe1JPmZayjCUoHWYx6gzGc33QEB6N3D0rBoEvAxp5atdQNhkJZSNIzmhdRpFx6ADU2PXYB6AjaP9Boi4/9sVedDHZ+KoxK3bJfbJM7aCvOG9ujOz8RvcO9LU9GCwTTKs1C9IHstGBeSfIepw6Sh+GjWARFAM5BLpUzZj2D8BtJG63XCCCBet0YhE8ZKELF89S99jyaP4iCaxShg6j3mHeX2+3BRxgAdeCZeZFlHlbWjEvaRxfNmxC4W83Df9+wdxQZPKjgTMUTrco8DcMnaCzBt3ry5qm/zE+FSw6vQOuqwPfI41WpGkV2g2J5NK41PMRE4MMKjE2YzQFdT88OlT0cxvE8S9rjrCxF7EqqiDUxePJGw97puGpaWcC2egZL6ZLV10U3BfJB7O6rPIlhA8PQqeWsto2/l8kqYtO4AU37cKKGukkbTIehB2uqmyiK0F54DyRtsZFKoUpUEo31voyX6I2k5aJXcIkuzuBLCUjheayaDU/qLdigXtJQM6sqdCOjJjagidFBBmiAw1E6jGS7xrO1McSmeX7MuQEVURBDqTNGBXhSgJ08kUlIw4IeXA+sZRgRUQowKj1jsx3RpIZjS9ewKauGgajlS2CN6dQly0bvdlAdMZmNgUUhI4eKHACucrB9d+qwCOo64tAHtyioESeJAvzq8nICvSiU4iiOgwI1cpfWSEqtFsNB6jXoHdfRIFU2zQydzrIEdCCN/WjOdrX3E6mq+/GUxfCIsCZq3SQshILuWvRR+V/Ann5HN6zabCJ0S6ObJE6jwHHsjvuoX1gG0J7aeoAGWumAPNBA4SiT8JRRqsWTOJdpGRR0fOYRjzVfbFU8UVTDP7HUYIY9qcPHzGhT7CELHB4uS6OJxywE2+F3yyqhpgQzBY1GoCo8Omd/vXmpbBqqTFWFQ5H3gujJ09Lf80Fdxmm5qKkee3vwfhpdvpc+84YR0E6Y5oOzbxAoHuOnEku1vmRvZ4kcWraWylsmMFlkKgq1dm1OtVEI+92kUYvuDKmulkuJNwkkqpCGQWNBV7v+vcAuARpaC+oGPzW9sPFr0S35ubvEB0IkPtd1XrCj3s1WhUKvC0HzuhbFXW0XSBG1B9hNYlV6+6RMYJubA31Hro3fNQ6PzX5BER2RjdcI06VOjPseHaayeA5A9tnRs8sd2AcI604IDPkyuPOM3BeJT3xf4QWCa3Rfxxs3gCAqos5FlU28yCe4q4lcXk2luB4KxqdBv4/6JBjICaLKdT02gwCLL0VdiWZhq1WsZHd2TOkGMtGxBebw+HowYDI3osfaKy020rDMSgnoD6vA5ROq01yC9BbBwwuscBZ+u5W+CyCwTqumXGf0VF1nKbdcb2u73vvH1ccR2JvqVktA+w130lrTwNDP1fbNQTStUJQpxMbtNx+Cn7sk2zmAo+sGgr5pQdD2o92jwuqQmeN82CmvrSPWz50POeNdBF6bLV7T/jYP3XH2D9qUmcPrV0fy/hMxPM0+k49mGR/CLloO4ydzigNc4uflEAdQqLrqt9HH2GIfxScOIk3NPz6VR9yOj7fzjp/MH9bvx8Zartr4uH4vQvJBYSN96x+a/Rij82W/WkcDOMbr8F5msv5wZmLiFH41fGTPWYy34C7rRaAbkjHOu9Iwwusrbn2LXtvOVZG/Ar3KuZzPLQLb6+fnW7p3lwpGLjuWtW7fZNfkX6Cm51i398hM4hSYSZJE06FQ13Geow/N3Ohbg/NaGSfiOTpF0XGZlL5aXWFT5UKxit+Cmjfod8WZf6Y3SZXbhKz4H90GL9BGtjcUbnVLB6U9/JchWe00R6+Kbq3v1+zWO4jg2sh2aCTq/D88lN28faSq9d5QDSF5pNWfxqCX56eXg9NqworHbO9njT54dnJ59qzR7T2DG3I6MPoQ/63bRq86tg5v+vEt5c1ClhSzUJ/PUDh1sADFq2CPalOh2sSk0EAY2UK+NQ5faa6qbV+GbD2mNey26VvDkGaUUhSIHqS+YXW1xqr5mTsBqQz2yvLaI85G+Gq4WtPl1hVhxbvQs4lwfLK/8YLEdY4sgmGzduRyfIhPt/kWu/aIX1tsCjlyxe81D9llb2z40bBs+ZEfEAODXCRlMzOEMnL0EzfUD3smKvat5AY2JB+YLtVTxQnxKsa2Gw8W2DRwoMBHueA2R5Ebn/jig7dx7j7I6UqFb8pp4wPae75FX17DTxU8gvIbikRo5e9sPs+cH3H176pYgR/vE9Lftz15poi0lvuI2/ffCfdHxOARbeP4nefUko6cRxxTUS8N5q8msoRB3MXI2aCkhe8tfutpwyP/Ik9JMEAGyCuEGvqBFRGgqIkQGDw7Pj3pO63re0tXfsGZ8RMsdpRKmI/cLNAF5fbrmAMDEXduQPsKVFt1UFEZbsp4cq3cAm873DM4hJEVgzRSY9oBZR12szfNAZ3rVxxgRs57+/qvOdpWt6cAQA58K5Fv2HGnuz3ipds7AZHc71cVSTQnPSZLSxIuiBrZJHBWOUbfJdGMbp/Qi5Ol2gFswwTCTECOvjJhNX/7P6EjANdKXH6FzzgXYEi0LvdyUKFDPPdLvLkME7kFynHrYiXXEXxrd+A0j4PBmZ6xFrGI78gzSQPQusFNASwlnKi1y+4RSzvQan1TvbMELoUUWh5IBwWF5YZENjdb1F1v0F+l/Zc/4LCFO4NuszhKpkgzQHOgYtI8Rv2xfx1tUX+rqfbGp8ku2PW5V479qHNjzTdFRvq8vealjFPDQyXeMphIQP+ymK8wpPJbfCrcaaQmRUzRg0EYTrNJGKLft1gS3YXkfAuqzt/Jmy/qDi+iJP/SNNWnn/tyOg2lHsR1TLAozE9r2oFTlSwAQOCwV7gO39T34yBiqjAb2EEQDQZDWgYBXLVH0MGou0MAQTQj69RhiKT478KsAloryFrSVrG9pbxKosNQmQZssHVAbMtsMdiVls5hZK1hsYcHQxlkjaQjbathoBqjxnk0AwVRJffptBGW0kiYKX8pr6NpXChX+tCTPLGqDLNrjnpqBFVZQVfS12FX1AADhYNDt4LShyIyds2lQaxgQFeXkyTRrss1enaIlNpuC6RP5+R5DWA0X1VVNuAt2Mm6MLcbbZch0ufn1hmaKgLq8uUIL7nmPrCKrmjch+iddABVw5yoLc71zchOXz3B1t48dogtCApYTzKxwaBOg4uy3cboMe421707JrY/MKTWkuxBGhrUrukDKkMxus+B8OPh4ES9E1igdSMoeqRLsAk8nulH0oN0tVEwsdzS3qD26dm7z0F73RHpM4e9MNRBW6/3x++O+JGM2/u1ElIZkIdWjILseKBxIlyrUMZ+jpIUWS5gRBgigw9DuhYIQ2TAYahvBpgb3yWI/Qo+/l3+513+Z5X/efL4+NGJfz7oPz5/dEfe/wqfPAZBcaSzocL18c+RCHp7/ufx6cXJOed/XgzOzs+h3eD0+OLiLv/zn5v/+S1iBmmOqKWQSW+MXzJtJJiZV5i0wnkxhU4r+8sKFJNyKyaLaHLtdzrPKKZWK/y+f2Ql3rlW1hplfZZo2mNQBqjwHDRXRCsVYfjRRCZoW3OwJcfsoxlPJmFPvNZTAN0/wSi5KpCZRkW412mGTr9MxZx/qL0NmIbIUZ2k1vvieZ0TEJcUCUsaD2XJ1WHreRHhfQS6KA1EF9ZS1o8PBT2voySbwHZ4FJssKf1ALREMataUyEjwJxIzD2AY2KtCoGlSh5q+WS1zISdFplS9ep/W/d/2Zg8BPqVXxaqOeMXYUJDsU4ezG7B/bTJyIoTiDFi1ROc9rk+mVTosaLQJxaK7HHuPhyLFE3HaF5976B6krlfxHKftPhXHZ6ZijlnCy1gpgz1LzudCGIgkRckmYbUizOsE01Do8G9AglUy1bGQV5RkCoZ6FKW+eL2IwCpB9KAAb8QP9Hos4ukUqmljXmFoOkydVHQ11DmdYJcEdV4nltzIlErg2xfPqlhNitJsCdHkqO9eDxPXEBpZOCA6hZJbxcnEMt1WGWKljlzXwcPyluw3Rr/OXoIqs+TdHNUQdvxD1tQ21T//+dBd16FUVzutFYNRd5Ixu+gi6lKmUZfyg7oUloyGB2D99ifkXEJnbZamKipK9FI2jJnd9CHzLK8UfrthiM6GMPSA/BzfR9vGzJ13Fwe6xJjvNPuLHIrnp/3jaiatCZ67bTHo/PK755cY0vlMYIT3qQ6i36Eym3zWiukKKzBsC1gWXdwypXa+ufxTA+axf6Zhclpj3RT4Yaazv9VNlKD/eyIxq0e8zTK08dGGRY6To2WqPAL9zVevXn318vc02eMzE/MPliknSWVpj+mspkqTbW3YRZUi1d2b0Xuy1AB7jr9o8YYhggClLxF9j4j9IYYD4rNgOGn0OZlavbAtFEBLps2q6Twq6+RpclhSIggMD+xM/bbi1Apz2yMGhilUoY7vC/NNFXlt/ECX/v7CPjXdTq/i75xw95u2hLu9ONCQrp/0HTmQruLg+iqeswrftm6hpren6zGfDiqq0XHVjV3ASPGRQ/4/ihGmm8Wx54045XMv2rowCQ7qL0D8mDIByBxiuhq/FMLjdK0upXkdcdM8rueUhpNyowNYdgO+b5sY7le0KTkutRkS3rGSGHWzEMMudfgnhorSoD0xaKYOk6KETJ8EdKmiBEhtHZncJkr4B953pXPDaQh/lU8xQQ+2O4ApasTEQOsdRMVLIZlmIMCzlTKBGdSU0WO/zurbFvtA1UDQ8UoFBXriMa47oBCMRnxFHcy7Hg2GuF/KFCgqgJJYnUwtn+on5HVSjCztsLuEETbBwO/XDOU5Zt5QhCtH01b5Oy4sQL9+AEggK4ABEjehHbcTluitHFXGl/twcNQbaHZiuNWwgtqn7J4mIGIeJiPNt9gVvtQhR6kHTJI5s1dDwl4c04HzgA0y6p4q95L46mH2X34xy+KE1K2KX+lkmd3g5E6VwtAaz2zCojG4ZaNjQDDbcTu6Htu3Sxqp8Fwdby/7J9pQGk4XmKUd0swLlSkDXALRXYOOfMJ5P5stBjYNPExwija76U2c0QT/Rpgwi9997YumXQkEoEOnyQ/biMGea8yZKtYutPZo2RLDCqFVtQq8pWUek++tw5pmMxmJljbJFMZpI/zpiPK7mN+ho9YUw2JRjg92UptwnynEG6FrQimyEvkFhpOD1TaPU2qi1z3RfN6M28Oqbj0gPVeZgoAmPQ3EHrVKoBlNaHKcTNgDyPoJj0hZVQ/FpK4aVymQUK6B65kbxt7I3v77JlfrJXxocrWVUx4i5empdVmfp5h9ZpNYEuU1Q/qSjNAdG9R08uvk09eUu2vSdikDPbDSzzka43XXI6WxJm6dI767Y1Sn89TpmNW1T1yaKrYw281GJ1XPixjQaggt+8Mfxm1Z5K/3c8dJc6UmmBGu3NdcTFZuYCed1tvDtKOTHBO7s062oFcWgR7K64pTEEc09Y17jkxkAJijhTr84s0WDx6YVA3KDKawfApEeO3ZtzTRlAJUAf5DGhyDKDN6GlihaHJK7OOsBv+Q1O0HdKTYaYCWfGZnA2yA2jcDZAEpKyXwjOONiBAAJAk9rAOi+qGue0jDPaRswirXHvpu22ANWmC9qOvaYAH3g2k9AdLtI7/Y0u9tv6kK8h7Amt+feocYNALRAyC2A1hyf7gZWIHBEaW2LpGIXHezGTWaweT1sj06MJiqu922txnoNh7OF5dVn7CJQMbBWNezV5uKJxppPmmNE1sCjHZXMMIxrQz/7S3V40bo8qhEZWZSR/0A4XBZau3eTcgk4U5gHxBPUYNtQVFNOLo9rLrkO1l4sjIAiYzqIRt0VceWdmkujfw48o/spLF2TRYNug70QzjLFb2ZRRPZz5KL+oEJprfkqv5a8041U9eykNg6qNdWA4WM8wc0gsCuit+Csjc6kN6P5ghmJWIaYP0SP37zgVFp39ZLUBMN+PRRv/napTbAlvcW/V21qxedC4glAAXfenSDOkvT2GCV1TI8nHFT67r6GM2VTVJ6LUildoEo0y8QgYfFNsdIQvg1jWcz9wrlFekmfc9/jZEW+v1MGBzG7uDqBTLac9NACEMjQyHet0e84TpYRaIril/xx6PQaw2NC5Z8IqtkpRpoxqKJtgtGhG06qsffS0dGGiY8njk/Uqt3IX2/o7z2mhmiFauthn2ztgn1tpRIgjPeOQjaW1Yf9tQpTlFGrRnTlA+oVm2QwIqgLsRe4VfLTk+LzLyLBvhhw91Kg+8mYFsKNQywa3XWs7h3wDV20GUtXDRLJXvOMNSQnftgTmPk4zZq5Bqu6CDs94fAUkmZ9hocP7Saoupgva6HFUiSvviSDHzyZUovB0TZ0aslY5LzVtYA+d0TNXx8bQV3M4dTd4aV8A6hrpUjx3kKvKIH8r7SpzADiGqe7JcOsP2LtvaDZvsa/3KyIF1rfg91QpE1GZqttitrrq6vIjBDv8d7MhoiPCPFEQjBp+7GpjQSHEFPbG6iWQmhH28cSfB2fbX9HPrWGMAzcWGXfv0aEzSh6zen1Cux25h3ygG9TQ7L0d0U5Eu/unmzTEabCrzd1xPVCBoDV092Bjg8bZTXybShtiEN4ITac8/Y7Km3nF/sUvvD8S14VeVT0XSW4zWTPmeu0s5ur2U/YGb7t2SH34jQSFH9xbwe4ZYRql00vc0zw9D7qCv1C1K4jN8sdhgyKjUB+a2nq2Wu3NFIj8BJUvxw1SVHCSKExepMHCMqV/zGQ7TvQV/ZbHfzyO13PexGAralh39gruZemqZOkuiKT0nZ1PjyK8zcdJzOx2Vzf1Km5odkdP8zsrnTimUE1a613Lk3ju5WcPrNVC3QzDurPgIY2n8tkOgVWztgPi539BecM2ri339xCaMfgoPYtcYhfiIkcJxDGaa/gIwKvAN+X4qEpJCCwOETt2BgzDrnBTACNNMC9sGhefAR8MhzEhiXcOTPfTvI4vAwaLF9zKxzxYE7smE3qhar8ZNTPz423+IfkllBYSuB+BFEFF0vckrjtWYA7sR4egKHayZYDOuBbjq5lHMzLA/VnJs0ZLwWUjpDJRDzGm7DPt5LkWi8SqrSjXNlXq00253gjEeHJs2kRpzmw6DNHWfeLtXwxjHTBCg669Cr3A8IHydQ6U/ez5Jacul/SAZIW/bHobQNbw/uz5apcf4RaRmqLNzd1AzvYG7GB6RmiDpuhyvMI9Zp3q1r9BNWkC+XS/EnFDl3GRp3n7vP3efuc/e5+9x97j53n7vP3efuc/f52M//A1U1XZoAeAAA")), mode="r:gz").extractall(SRC)
sys.path.insert(0, SRC); sys.path.insert(0, SRC + "/pilot")
import analyze_v2
metas = glob.glob("/kaggle/input/**/meta3d.json", recursive=True)
INPUTS = os.path.commonpath([os.path.dirname(os.path.dirname(os.path.dirname(p))) for p in metas])
ITEMS = analyze_v2.load_items(INPUTS)
SEL = "all"
if SEL != "all":
    ITEMS = {k: m for k, m in ITEMS.items() if any(s in k for s in SEL.split(","))}
print(len(ITEMS), "items:", ", ".join(sorted(ITEMS)))
NAME, ENDPOINT, RES, DUR = "seedance", "bytedance/seedance-2.5/image-to-video", "480p", "5"
SAMPLES, USD_PER_S, MAX_USD = 1, 0.27, 45.0
OUT = f"/kaggle/working/out/{NAME}"; RAW = OUT + "_raw"
os.makedirs(OUT, exist_ok=True); os.makedirs(RAW, exist_ok=True)
SCENE = "a curved metal tube, filmed from above by a static camera"
# one prompt for every item: it says the ball is about to exit, never which way it heads
# (the LTX run named a compass direction instead)
PROMPT = (f"A glossy red ball with an orange stripe rolls quickly through {SCENE}. The ball is exiting the tube: "
          "it rolls out of the open end of the tube and keeps rolling. "
          "Realistic, smooth continuous motion, static camera, the ball keeps its size, shape and colour.")
print("prompt:", PROMPT)
print("estimated cost:", len(ITEMS) * SAMPLES, "clips x", DUR, "s x $", USD_PER_S, "= $", round(len(ITEMS) * SAMPLES * float(DUR) * USD_PER_S, 2))

In [ ]:
# generate: one call per clip, saved as it goes; stops before the spending cap
def call(args):
    eps = [ENDPOINT] + ([ENDPOINT.replace("seedance-2.5/", "seedance-2.5/us/")] if "/us/" not in ENDPOINT else [])
    last, seed = None, args.pop("seed", None)
    for ep in eps:
        for extra in (dict(resolution=RES, duration=DUR, seed=seed), {}):
            try:
                kw = dict(args, **{k: v for k, v in extra.items() if v is not None})
                return ep, kw, fal_client.subscribe(ep, arguments=kw, with_logs=False)
            except Exception as e:   # unknown endpoint or argument: try the next form, print why
                last = e; print("  call failed with", ep, sorted(kw), "->", str(e)[:300])
    raise last
spent, log = 0.0, []
for iid in sorted(ITEMS):
    m = ITEMS[iid]
    url = None
    for j in range(SAMPLES):
        path = f"{OUT}/{iid}__still__s{j}.mp4"
        if os.path.exists(path): continue
        if spent + float(DUR) * USD_PER_S > MAX_USD:
            print("spending cap reached, stopping"); break
        if url is None:
            url = fal_client.upload_file(os.path.join(m["dir"], m["files"]["still"]))
        t0 = time.time()
        ep, kw, res = call(dict(prompt=PROMPT, image_url=url, seed=7919 * j + sum(map(ord, iid))))
        vurl = (res.get("video") or {}).get("url") or res.get("url")
        raw = f"{RAW}/{iid}__still__s{j}.mp4"; urllib.request.urlretrieve(vurl, raw)
        # bring the clip back to the stimulus size so the tracker's coordinates match; keep the model's fps
        rd = imageio.get_reader(raw); fps = rd.get_meta_data().get("fps", 24); fr = [f[..., :3] for f in rd]; rd.close()
        H0, W0 = fr[0].shape[:2]; W, H = m["size"]
        if abs(W0 / H0 - W / H) > 0.02: print(f"  note: aspect changed {W0}x{H0} vs {W}x{H}; stretched back")
        imageio.mimwrite(path, [np.array(Image.fromarray(f).resize((W, H))) for f in fr], fps=fps, macro_block_size=16)
        spent += float(DUR) * USD_PER_S
        log.append(dict(item_id=iid, sample=j, endpoint=ep, args=sorted(kw), fps=fps, n_frames=len(fr), size=f"{W0}x{H0}", seconds=round(time.time() - t0)))
        print(f"{iid} s{j}: {len(fr)} frames {W0}x{H0} @ {fps} fps, {time.time()-t0:.0f}s, about ${spent:.2f} so far", flush=True)
json.dump(log, open(f"{OUT}/calls.json", "w"), indent=1)
FPS = log[0]["fps"] if log else 24

In [ ]:
# score like the LTX run: frame 0 is the still, the tracker follows the ball after the exit
res = "/kaggle/working/results"
r = subprocess.run([sys.executable, SRC + "/pilot/analyze_v2.py", "--inputs", INPUTS, "--gen", f"{NAME}={OUT}",
                    "--cond", f"{NAME}_still=1", "--fps", f"{NAME}={FPS}", "--vjepa", "/nonexistent.csv", "--out", res],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
for f in [f"{res}/summary.csv", f"{res}/per_clip.csv"]:
    print("=====", f); print(open(f).read() if os.path.exists(f) else "(missing)")
!cd /kaggle/working && zip -qr slide21_api_results.zip results out && ls -la slide21_api_results.zip